'Validação' = consultas de conferência dos resultados

# Evidências de Execução

Esta seção reúne as 3 evidências do entregável: carga inicial, reexecução
idempotente (sem novidade) e execução incremental com novidade.

## Evidência 1 — Carga inicial

Primeira execução bem-sucedida de cada coleção. A Coleção users foi executada bem sucedida em um momento diferente das demasi devido à uma correção da arquitetura sobre schema drifts que acabou interrompendo a carga na primeira tentativa que já tipava o schema a ser recebido.

In [0]:
%sql
WITH primeira_execucao AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY collection ORDER BY start_time ASC) AS rn
  FROM meu_catalog.bronze.control_ingestion_log
  WHERE status = 'SUCCESS'
)
SELECT collection, load_type, qtd_lida_origem, qtd_gravada_destino, start_time
FROM primeira_execucao
WHERE rn = 1
ORDER BY start_time;

## Evidência 2 — Reexecução idempotente

Pipeline rodada de novo sem mudança na origem: qtd_gravada_destino = 0 em
todas as coleções, provando que o MERGE insert-only não duplica.

In [0]:
%sql
SELECT collection, load_type, qtd_lida_origem, qtd_gravada_destino, start_time
FROM meu_catalog.bronze.control_ingestion_log
WHERE _ingestion_id IN (
  'a53f560f-c841-4ae6-96c4-d85785ac426f',  -- users
  '913837db-47f9-4db8-b830-8932b6c7f4b5',  -- theaters
  'e8825970-4578-4e8a-9b99-0627c9bbf2a4',  -- sessions
  '7987f4f0-d1cd-4297-8f42-cd3719358c76',  -- embedded_movies
  'b5a1f3c1-1344-47f6-8653-bf3a6ce7ad05',  -- movies
  'a34c206e-1183-4bfe-8b4e-1a990bad140f'   -- comments
)
ORDER BY start_time;

## Evidência 3 — Incremental

Teste controlado: um
documento existente foi removido da Bronze e a watermark retrocedida
manualmente na control table (linha 'teste-simulacao-incremental'),
simulando a chegada de um documento novo. A execução seguinte capturou e
gravou esse documento de novo.

In [0]:
%sql
SELECT _ingestion_id, collection, status, qtd_lida_origem, qtd_gravada_destino,
       watermark_final, mensagem_erro, start_time
FROM meu_catalog.bronze.control_ingestion_log
WHERE _ingestion_id IN ('teste-simulacao-incremental', 'c024d08c-687d-4641-b789-03e0bbd96c80')
ORDER BY start_time;

In [0]:
%sql
SELECT * FROM meu_catalog.bronze.users
LIMIT 10